# Official MedMNIST ResNet-50 28x28 Metrics

This notebook evaluates the official MedMNIST experiment predictions for PathMNIST ResNet-50 at 28x28. It does not train a model. It downloads `predictions.zip` from the MedMNIST experiments Zenodo record, extracts the official PathMNIST ResNet-50 test CSVs, loads the official PathMNIST test labels through `medmnist.Evaluator`, converts scores to class predictions with `argmax`, and computes the same metric family used for the local baseline reports.

Official source: Zenodo record `10.5281/zenodo.7782114`, file `predictions.zip`. The archive lists three PathMNIST ResNet-50 28x28 test files:

- `pathmnist_test_[AUC]0.991_[ACC]0.908@resnet50_28_1.csv`
- `pathmnist_test_[AUC]0.990_[ACC]0.902@resnet50_28_2.csv`
- `pathmnist_test_[AUC]0.989_[ACC]0.924@resnet50_28_3.csv`

The headline benchmark AUC/ACC is obtained from these official prediction files, not from local retraining.

In [1]:
from pathlib import Path
import json
import re
import sys
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from sklearn.metrics import fbeta_score
from medmnist import Evaluator, INFO

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from pathmnist.metrics import report_dict

DATA_FLAG = "pathmnist"
SPLIT = "test"
PREDICTION_URL = "https://zenodo.org/records/7782114/files/predictions.zip?download=1"

DATA_DIR = REPO_ROOT / "data"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "medmnist_predictions"
EXTRACT_DIR = ARTIFACT_DIR / "predictions"
PREDICTION_ZIP = ARTIFACT_DIR / "predictions.zip"
RESULT_DIR = REPO_ROOT / "results" / "official_medmnist_resnet50_28"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Prediction archive:", PREDICTION_ZIP)
print("Result directory:", RESULT_DIR)

Repository root: /Users/nick/Workspace/pathmnist-classification-improvement
Prediction archive: /Users/nick/Workspace/pathmnist-classification-improvement/artifacts/medmnist_predictions/predictions.zip
Result directory: /Users/nick/Workspace/pathmnist-classification-improvement/results/official_medmnist_resnet50_28


## Download And Extract

`predictions.zip` is about 2 GB. If the file already exists at `artifacts/medmnist_predictions/predictions.zip`, this cell will reuse it.

In [2]:
def download_predictions_zip() -> Path:
    if PREDICTION_ZIP.exists():
        print(f"Using existing archive: {PREDICTION_ZIP}")
        return PREDICTION_ZIP

    print(f"Downloading {PREDICTION_URL}")
    urllib.request.urlretrieve(PREDICTION_URL, PREDICTION_ZIP)
    print(f"Downloaded: {PREDICTION_ZIP}")
    return PREDICTION_ZIP


def extract_official_pathmnist_resnet50_28_test_csvs(zip_path: Path) -> list[Path]:
    pattern = re.compile(r"^pathmnist_test_\[AUC\].*_\[ACC\].*@resnet50_28_\d\.csv$")
    with zipfile.ZipFile(zip_path) as archive:
        names = sorted(name for name in archive.namelist() if pattern.match(Path(name).name))
        if not names:
            raise FileNotFoundError("No PathMNIST ResNet-50 28x28 test CSVs found in predictions.zip")
        extracted = []
        for name in names:
            out_path = EXTRACT_DIR / Path(name).name
            if not out_path.exists():
                out_path.write_bytes(archive.read(name))
            extracted.append(out_path)
    return extracted


prediction_zip = download_predictions_zip()
official_csvs = extract_official_pathmnist_resnet50_28_test_csvs(prediction_zip)
official_csvs

Downloaded: /Users/nick/Workspace/pathmnist-classification-improvement/artifacts/medmnist_predictions/predictions.zip


[PosixPath('/Users/nick/Workspace/pathmnist-classification-improvement/artifacts/medmnist_predictions/predictions/pathmnist_test_[AUC]0.989_[ACC]0.924@resnet50_28_3.csv'),
 PosixPath('/Users/nick/Workspace/pathmnist-classification-improvement/artifacts/medmnist_predictions/predictions/pathmnist_test_[AUC]0.990_[ACC]0.902@resnet50_28_2.csv'),
 PosixPath('/Users/nick/Workspace/pathmnist-classification-improvement/artifacts/medmnist_predictions/predictions/pathmnist_test_[AUC]0.991_[ACC]0.908@resnet50_28_1.csv')]

## Load Official Labels

`medmnist.Evaluator` loads the official labels from `data/pathmnist.npz`. The local `data` directory must contain the standard PathMNIST NPZ. If it is missing, download it first with MedMNIST or run one of the existing training/evaluation scripts once.

In [3]:
evaluator = Evaluator(DATA_FLAG, SPLIT, root=str(DATA_DIR))
y_true = evaluator.labels.reshape(-1).astype(int)

labels = INFO[DATA_FLAG]["label"]
class_names = [labels[str(i)] for i in range(len(labels))]

print("Test labels:", y_true.shape)
pd.Series(y_true).value_counts().sort_index().rename(index=dict(enumerate(class_names)))

Test labels: (7180,)


adipose                                 1338
background                               847
debris                                   339
lymphocytes                              634
mucus                                   1035
smooth muscle                            592
normal colon mucosa                      741
cancer-associated stroma                 421
colorectal adenocarcinoma epithelium    1233
Name: count, dtype: int64

## Compute Metrics

Each official CSV stores one row per dataset index, with the predicted score vector after the index column. For multi-class PathMNIST, the predicted class is `argmax(score_vector)`.

In [4]:
def load_prediction_scores(csv_path: Path) -> np.ndarray:
    df = pd.read_csv(csv_path, index_col=0, header=None).sort_index()
    scores = df.to_numpy(dtype=float)
    if scores.shape != (len(y_true), len(class_names)):
        raise ValueError(f"Unexpected score shape for {csv_path.name}: {scores.shape}")
    return scores


def metric_row(name: str, report: dict, medmnist_auc: float, medmnist_acc: float, y_pred: np.ndarray) -> dict:
    cancer = report["classification_report"]["colorectal adenocarcinoma epithelium"]
    stroma = report["classification_report"]["cancer-associated stroma"]
    stroma_label = class_names.index("cancer-associated stroma")
    stroma_f2 = fbeta_score(y_true == stroma_label, y_pred == stroma_label, beta=2, zero_division=0)
    return {
        "file": name,
        "medmnist_auc": medmnist_auc,
        "medmnist_acc": medmnist_acc,
        "auc": report["auc"],
        "acc": report["acc"],
        "precision_macro": report["precision_macro"],
        "recall_macro": report["recall_macro"],
        "specificity_macro": report["specificity_macro"],
        "f1_macro": report["f1_macro"],
        "f2_macro": report["f2_macro"],
        "cancer_precision": report["cancer_precision"],
        "cancer_recall": report["cancer_recall"],
        "cancer_specificity": report["cancer_specificity"],
        "cancer_f1": report["cancer_f1"],
        "cancer_f2": report["cancer_f2"],
        "stroma_precision": stroma["precision"],
        "stroma_recall": stroma["recall"],
        "stroma_f1": stroma["f1-score"],
        "stroma_f2": stroma_f2,
    }


reports = {}
rows = []
predictions = {}

for csv_path in official_csvs:
    y_score = load_prediction_scores(csv_path)
    y_pred = y_score.argmax(axis=1)
    medmnist_metrics = evaluator.evaluate(y_score)
    report = report_dict(y_true, y_score, class_names)

    reports[csv_path.name] = report
    predictions[csv_path.name] = y_pred
    rows.append(metric_row(csv_path.name, report, medmnist_metrics.AUC, medmnist_metrics.ACC, y_pred))

summary = pd.DataFrame(rows).sort_values(["auc", "acc"], ascending=False).reset_index(drop=True)
summary

,file,medmnist_auc,medmnist_acc,auc,acc,precision_macro,recall_macro,specificity_macro,f1_macro,f2_macro,cancer_precision,cancer_recall,cancer_specificity,cancer_f1,cancer_f2,stroma_precision,stroma_recall,stroma_f1,stroma_f2
0,pathmnist_test_[AUC]0.991_[ACC]0.908@resnet50_...,0.990986,0.907521,0.990986,0.907521,0.896409,0.881007,0.988478,0.878751,0.878350,0.913011,0.978913,0.980663,0.944814,0.964982,0.929204,0.498812,0.649150,0.549738
1,pathmnist_test_[AUC]0.990_[ACC]0.902@resnet50_...,0.989713,0.901950,0.989713,0.901950,0.886857,0.874415,0.987817,0.870906,0.871264,0.913277,0.965126,0.980999,0.938486,0.954290,0.916667,0.496437,0.644068,0.546548
2,pathmnist_test_[AUC]0.989_[ACC]0.924@resnet50_...,0.989448,0.923816,0.989448,0.923816,0.908861,0.898120,0.990529,0.897066,0.896568,0.936371,0.966748,0.986380,0.951317,0.960516,0.912351,0.543943,0.681548,0.591731


## Aggregate Official Runs

The official archive contains three ResNet-50 28x28 runs. This cell summarizes the mean and standard deviation across those runs, plus the best run by `AUC + ACC` for a single-file comparison.

In [5]:
metric_columns = [col for col in summary.columns if col != "file"]
aggregate = pd.DataFrame({
    "mean": summary[metric_columns].mean(),
    "std": summary[metric_columns].std(ddof=1),
    "min": summary[metric_columns].min(),
    "max": summary[metric_columns].max(),
})

best_idx = (summary["auc"] + summary["acc"]).idxmax()
best_file = summary.loc[best_idx, "file"]

print("Best single official ResNet-50 28x28 test file by AUC + ACC:")
print(best_file)

aggregate

Best single official ResNet-50 28x28 test file by AUC + ACC:
pathmnist_test_[AUC]0.989_[ACC]0.924@resnet50_28_3.csv


,mean,std,min,max
medmnist_auc,0.990049,0.000822,0.989448,0.990986
medmnist_acc,0.911096,0.011363,0.901950,0.923816
auc,0.990049,0.000822,0.989448,0.990986
acc,0.911096,0.011363,0.901950,0.923816
precision_macro,0.897376,0.011034,0.886857,0.908861
recall_macro,0.884514,0.012235,0.874415,0.898120
specificity_macro,0.988941,0.001414,0.987817,0.990529
f1_macro,0.882241,0.013425,0.870906,0.897066
f2_macro,0.882060,0.013054,0.871264,0.896568
cancer_precision,0.920886,0.013411,0.913011,0.936371


## Per-Class Metrics For Best Official Run

In [6]:
best_report = reports[best_file]

per_class = pd.DataFrame(best_report["classification_report"]).T
per_class = per_class.loc[class_names + ["accuracy", "macro avg", "weighted avg"]]
per_class

,precision,recall,f1-score,support
adipose,0.986411,0.922272,0.953264,1338.000000
background,0.891579,1.000000,0.942682,847.000000
debris,0.805851,0.893805,0.847552,339.000000
lymphocytes,0.967742,0.993691,0.980545,634.000000
mucus,0.973294,0.950725,0.961877,1035.000000
smooth muscle,0.747076,0.863176,0.800940,592.000000
normal colon mucosa,0.959072,0.948718,0.953867,741.000000
cancer-associated stroma,0.912351,0.543943,0.681548,421.000000
colorectal adenocarcinoma epithelium,0.936371,0.966748,0.951317,1233.000000
accuracy,0.923816,0.923816,0.923816,0.923816


## Confusion Matrix For Best Official Run

In [7]:
confusion = pd.DataFrame(best_report["confusion_matrix"], index=class_names, columns=class_names)
confusion

,adipose,background,debris,lymphocytes,mucus,smooth muscle,normal colon mucosa,cancer-associated stroma,colorectal adenocarcinoma epithelium
adipose,1234,0,1,0,18,81,2,0,2
background,0,847,0,0,0,0,0,0,0
debris,0,9,303,0,0,19,0,7,1
lymphocytes,0,0,3,630,0,0,0,0,1
mucus,17,26,1,0,984,0,4,0,3
smooth muscle,0,62,5,0,1,511,0,12,1
normal colon mucosa,0,0,3,5,3,0,703,2,25
cancer-associated stroma,0,6,55,5,3,73,2,229,48
colorectal adenocarcinoma epithelium,0,0,5,11,2,0,22,1,1192


## Save Results

In [8]:
summary.to_csv(RESULT_DIR / "official_resnet50_28_test_summary.csv", index=False)
aggregate.to_csv(RESULT_DIR / "official_resnet50_28_test_aggregate.csv")
per_class.to_csv(RESULT_DIR / "official_resnet50_28_best_per_class.csv")
confusion.to_csv(RESULT_DIR / "official_resnet50_28_best_confusion_matrix.csv")

payload = {
    "source": {
        "zenodo_record": "https://zenodo.org/records/7782114",
        "predictions_zip": PREDICTION_URL,
        "data_flag": DATA_FLAG,
        "split": SPLIT,
        "method": "resnet50_28",
    },
    "summary": summary.to_dict(orient="records"),
    "aggregate": aggregate.to_dict(orient="index"),
    "best_file": best_file,
    "best_report": best_report,
}

(RESULT_DIR / "official_resnet50_28_test_metrics.json").write_text(json.dumps(payload, indent=2))

print("Saved outputs to", RESULT_DIR)

Saved outputs to /Users/nick/Workspace/pathmnist-classification-improvement/results/official_medmnist_resnet50_28
